In [2]:
import pandas as pd
import numpy as np
data = {
    "Student_ID": [101, 102, 103, 104, 105, 105, 106, 107, 108, 109],
    "Age": [20, 21, np.nan, 22, 20, 20, 21, 200, 22, 20],
    "Gender": [
        "Male", "Female", "female", "Male", "Male",
        "Male", "Female", "Male", "FEMALE", "Female"
    ],
    "Study_Hours": [2, 5, 3, np.nan, 4, 4, 6, 2, 8, 3],
    "Marks": [45, 78, 55, 60, 72, 72, np.nan, 90, 150, 48],
    "Department": [
        "CSE", "ECE", "CSE", "cse", "ECE",
        "ECE", "CSE", "EEE", "ece", "CSE"
    ],
    "Result": [
        "Fail", "Pass", "Fail", "Pass", "Pass",
        "Pass", "Pass", "Pass", "Pass", "Fail"
    ]
}
df = pd.DataFrame(data)
print(df)

   Student_ID    Age  Gender  Study_Hours  Marks Department Result
0         101   20.0    Male          2.0   45.0        CSE   Fail
1         102   21.0  Female          5.0   78.0        ECE   Pass
2         103    NaN  female          3.0   55.0        CSE   Fail
3         104   22.0    Male          NaN   60.0        cse   Pass
4         105   20.0    Male          4.0   72.0        ECE   Pass
5         105   20.0    Male          4.0   72.0        ECE   Pass
6         106   21.0  Female          6.0    NaN        CSE   Pass
7         107  200.0    Male          2.0   90.0        EEE   Pass
8         108   22.0  FEMALE          8.0  150.0        ece   Pass
9         109   20.0  Female          3.0   48.0        CSE   Fail


3. Inspect the raw data
Before cleaning, understand the structure and quality of the dataset.

In [3]:
print(df.head())
print(df.shape)
print(df.columns)
df.info()
print(df.describe())
print(df.describe(include="all"))
print(df.nunique())
print(df.dtypes)

   Student_ID   Age  Gender  Study_Hours  Marks Department Result
0         101  20.0    Male          2.0   45.0        CSE   Fail
1         102  21.0  Female          5.0   78.0        ECE   Pass
2         103   NaN  female          3.0   55.0        CSE   Fail
3         104  22.0    Male          NaN   60.0        cse   Pass
4         105  20.0    Male          4.0   72.0        ECE   Pass
(10, 7)
Index(['Student_ID', 'Age', 'Gender', 'Study_Hours', 'Marks', 'Department',
       'Result'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Student_ID   10 non-null     int64  
 1   Age          9 non-null      float64
 2   Gender       10 non-null     object 
 3   Study_Hours  9 non-null      float64
 4   Marks        9 non-null      float64
 5   Department   10 non-null     object 
 6   Result       10 non-null     object 
d

4. Identify and handle missing values
A missing value means that a data item is unavailable or not recorded.
Examples include a student's missing age, marks or study hours.

Step 4.1 — Inspect missing values

In [4]:
from numpy._core.defchararray import count
print(df.isnull().sum())
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)

Student_ID     0
Age            1
Gender         0
Study_Hours    1
Marks          1
Department     0
Result         0
dtype: int64
Student_ID      0.0
Age            10.0
Gender          0.0
Study_Hours    10.0
Marks          10.0
Department      0.0
Result          0.0
dtype: float64


Step 4.2 — Handle missing values
There are several common methods:
Method	When to use
Remove rows	Only when justified and enough data remains.
Mean imputation	Numerical data without strong skew or extreme values.
Median imputation	Numerical data that may be skewed or contain outliers.
Mode imputation	Categorical data
Forward/backward fill	Appropriate ordered or time-series data.

In [5]:
df_clean = df.copy()
numeric_cols = ["Age", "Study_Hours", "Marks"]
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].mean()
    )
print(df_clean.isnull().sum())

Student_ID     0
Age            0
Gender         0
Study_Hours    0
Marks          0
Department     0
Result         0
dtype: int64


5. Identify and remove duplicate records
Duplicate records may cause some observations to be counted more than once.

Step 5.1 — Find duplicates

In [6]:
print("Duplicate rows:", df_clean.duplicated().sum())

print(df_clean[df_clean.duplicated(keep=False)])

Duplicate rows: 1
   Student_ID   Age Gender  Study_Hours  Marks Department Result
4         105  20.0   Male          4.0   72.0        ECE   Pass
5         105  20.0   Male          4.0   72.0        ECE   Pass


Step 5.2 — Remove exact duplicate rows

In [8]:
df_clean = df_clean.drop_duplicates()

print("Shape after removing duplicates:", df_clean.shape)

Shape after removing duplicates: (9, 7)


6. Identify and correct inconsistencies:
For example:
- CSE, cse, Cse
- Female, female, FEMALE
- Extra spaces such as " CSE "

Step 6.1 — Inspect categorical values

In [9]:
print(df_clean["Gender"].value_counts(dropna=False))
print(df_clean["Department"].value_counts(dropna=False))
print(df_clean["Result"].value_counts(dropna=False))

Gender
Male      4
Female    3
female    1
FEMALE    1
Name: count, dtype: int64
Department
CSE    4
ECE    2
cse    1
EEE    1
ece    1
Name: count, dtype: int64
Result
Pass    6
Fail    3
Name: count, dtype: int64


In [10]:
for col in ["Gender", "Department", "Result"]:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.title()
    )
print(df_clean["Gender"].unique())
print(df_clean["Department"].unique())
print(df_clean["Result"].unique())

<StringArray>
['Male', 'Female']
Length: 2, dtype: string
<StringArray>
['Cse', 'Ece', 'Eee']
Length: 3, dtype: string
<StringArray>
['Fail', 'Pass']
Length: 2, dtype: string


In [11]:
df_clean["Department"] = (
    df_clean["Department"].str.upper()
)

print(df_clean["Department"].unique())

<StringArray>
['CSE', 'ECE', 'EEE']
Length: 3, dtype: string


In [12]:
print(df_clean)


   Student_ID         Age  Gender  Study_Hours       Marks Department Result
0         101   20.000000    Male     2.000000   45.000000        CSE   Fail
1         102   21.000000  Female     5.000000   78.000000        ECE   Pass
2         103   40.666667  Female     3.000000   55.000000        CSE   Fail
3         104   22.000000    Male     4.111111   60.000000        CSE   Pass
4         105   20.000000    Male     4.000000   72.000000        ECE   Pass
6         106   21.000000  Female     6.000000   74.444444        CSE   Pass
7         107  200.000000    Male     2.000000   90.000000        EEE   Pass
8         108   22.000000  Female     8.000000  150.000000        ECE   Pass
9         109   20.000000  Female     3.000000   48.000000        CSE   Fail


7. Identify and handle outliers
An outlier is an observation that is unusually far from the other observations.
Examples in our dataset:
- Age = 200 years
- Marks = 150 out of 100
An unusual value is not always an error. For example, a student studying 10 hours per day may be perfectly valid. Always investigate the value before modifying it

Step 7.1 — Detect outliers using the IQR method
The Interquartile Range (IQR) is:
\[
IQR=Q_3-Q_1
\]

The commonly used lower and upper fences are:
\[
Lower=Q_1-1.5(IQR)
\]

\[
Upper=Q_3+1.5(IQR)
\]

In [ ]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower) |
        (data[column] > upper)
    ]

    print(f"\nColumn: {column}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower fence:", lower)
    print("Upper fence:", upper)
    print("Outlier rows:\n", outliers)

for col in ["Age", "Study_Hours", "Marks"]:
    detect_outliers_iqr(df_clean, col)


Column: Age
Q1: 20.0
Q3: 22.0
IQR: 2.0
Lower fence: 17.0
Upper fence: 25.0
Outlier rows:
    Student_ID         Age  Gender  Study_Hours  Marks Department  Result
2         103   40.666667  Female          3.0   55.0        CSE       0
7         107  200.000000    Male          2.0   90.0        EEE       1

Column: Study_Hours
Q1: 3.0
Q3: 5.0
IQR: 2.0
Lower fence: 0.0
Upper fence: 8.0
Outlier rows:
 Empty DataFrame
Columns: [Student_ID, Age, Gender, Study_Hours, Marks, Department, Result]
Index: []

Column: Marks
Q1: 55.0
Q3: 78.0
IQR: 23.0
Lower fence: 20.5
Upper fence: 112.5
Outlier rows:
    Student_ID   Age  Gender  Study_Hours  Marks Department  Result
8         108  22.0  Female          8.0  150.0        ECE       1


Step 7.2 — Correct invalid values
For this example, we know that marks must be between 0 and 100 and that age 200 is invalid for this student dataset. We will treat these as data-quality errors rather than blindly deleting entire rows.

In [ ]:
df_clean.loc[
    ~df_clean["Age"].between(15, 100),
    "Age"
] = np.nan
df_clean.loc[
    ~df_clean["Marks"].between(0, 100),
    "Marks"
] = np.nan
for col in ["Age", "Marks"]:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )
print(df_clean[["Age", "Study_Hours", "Marks"]])

         Age  Study_Hours      Marks
0  20.000000     2.000000  45.000000
1  21.000000     5.000000  78.000000
2  40.666667     3.000000  55.000000
3  22.000000     4.111111  60.000000
4  20.000000     4.000000  72.000000
6  21.000000     6.000000  74.444444
7  21.000000     2.000000  90.000000
8  22.000000     8.000000  66.000000
9  20.000000     3.000000  48.000000


/tmp/ipykernel_1368/1592470159.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = df_clean[col].fillna(


ther outlier treatment methods
Method	Explanation
Remove	Delete a confirmed erroneous observation when justified
Capping	Replace extreme values with chosen boundary values
Transformation	Apply log or another transformation where appropriate
Retain	Keep genuine extreme observations
Imputation	Replace confirmed invalid values with a justified estimate

8. Encode categorical data
Machine learning algorithms generally require numerical input features. Encoding converts categorical values into numerical representations.
Our dataset has three categorical columns: Gender, Department and Result.

Method 1 — Label or binary encoding
For a binary target variable such as Result, we can map the categories explicitly.

In [14]:
df_clean["Result"] = df_clean["Result"].map({
    "Fail": 0,
    "Pass": 1
})
print(df_clean[["Student_ID", "Result"]])

   Student_ID  Result
0         101     NaN
1         102     NaN
2         103     NaN
3         104     NaN
4         105     NaN
6         106     NaN
7         107     NaN
8         108     NaN
9         109     NaN


Method 2 — One-hot encoding
For nominal input features such as Gender and Department, one-hot encoding creates separate binary columns.

In [ ]:
df_encoded = pd.get_dummies(
    df_clean,
    columns=["Gender", "Department"],
    dtype=int
)
print(df_encoded.head())
print(df_encoded.columns)

   Student_ID        Age  Study_Hours  Marks  Result  Gender_Female  \
0         101  20.000000     2.000000   45.0     NaN              0   
1         102  21.000000     5.000000   78.0     NaN              1   
2         103  40.666667     3.000000   55.0     NaN              1   
3         104  22.000000     4.111111   60.0     NaN              0   
4         105  20.000000     4.000000   72.0     NaN              0   

   Gender_Male  Department_CSE  Department_ECE  Department_EEE  
0            1               1               0               0  
1            0               0               1               0  
2            0               1               0               0  
3            1               1               0               0  
4            1               0               1               0  
Index(['Student_ID', 'Age', 'Study_Hours', 'Marks', 'Result', 'Gender_Female',
       'Gender_Male', 'Department_CSE', 'Department_ECE', 'Department_EEE'],
      dtype='object')


9. Scale numerical features
Scaling puts numerical features on comparable scales.
Our numerical features have different units:
- Age: years
- Study_Hours: hours
- Marks: marks out of 100
Two widely used methods are StandardScaler and MinMaxScaler.
Method 1 — Standardization
StandardScaler transforms each feature using:
z=x-μ/σ



In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ["Age", "Study_Hours", "Marks"]

scaler = StandardScaler()

df_scaled = df_encoded.copy()

df_scaled[scale_cols] = scaler.fit_transform(
    df_scaled[scale_cols]
)

print(df_scaled[scale_cols].round(3))

     Age  Study_Hours  Marks
0 -0.491       -1.146 -1.463
1 -0.331        0.473  0.906
2  2.809       -0.607 -0.745
3 -0.171       -0.007 -0.386
4 -0.491       -0.067  0.475
6 -0.331        1.013  0.651
7 -0.331       -1.146  1.767
8 -0.171        2.093  0.044
9 -0.491       -0.607 -1.248


10. Compare raw data with preprocessed data

In [ ]:
print("RAW DATA")
print(df.head())

print("\nCLEANED DATA")
print(df_clean.head())

print("\nENCODED DATA")
print(df_encoded.head())

print("\nSCALED DATA")
print(df_scaled.head())

RAW DATA
   Student_ID   Age  Gender  Study_Hours  Marks Department Result
0         101  20.0    Male          2.0   45.0        CSE   Fail
1         102  21.0  Female          5.0   78.0        ECE   Pass
2         103   NaN  female          3.0   55.0        CSE   Fail
3         104  22.0    Male          NaN   60.0        cse   Pass
4         105  20.0    Male          4.0   72.0        ECE   Pass

CLEANED DATA
   Student_ID        Age  Gender  Study_Hours  Marks Department  Result
0         101  20.000000    Male     2.000000   45.0        CSE     NaN
1         102  21.000000  Female     5.000000   78.0        ECE     NaN
2         103  40.666667  Female     3.000000   55.0        CSE     NaN
3         104  22.000000    Male     4.111111   60.0        CSE     NaN
4         105  20.000000    Male     4.000000   72.0        ECE     NaN

ENCODED DATA
   Student_ID        Age  Study_Hours  Marks  Result  Gender_Female  \
0         101  20.000000     2.000000   45.0     NaN            